# Fresh Return-v4 — train C-R/C-N trước, benchmark ở notebook riêng

1. Tạo một Kaggle Dataset từ **BS5_RETURN_V4_KAGGLE_DATASET.zip**, rồi Add Input.
2. Import notebook này, chọn GPU nếu có rồi **Run All**. Không cần checkpoint/kết quả cũ.
3. Bốn run fresh: C-R/C-N × seed551/552, mỗi run60k. Lịch S1–S3 giữ150 slot/game, S4 dùng1.200 slot/game; không đổi reward, router, pin, trạm sạc hay Return-v4.
4. Validation0/6k/…/60k:5 trace cố định ×1.200 slot. Tổng chính:240k train +264k validation. Notebook này **không chạy benchmark10k**.

UE tape và tham số game dùng chung giữa hai nhánh cùng seed. Initial weights giống nhau trong mỗi cặp; optimizer/replay/exploration độc lập. C-N mask mọi action Relay, kể cả target-Q, và tắt Relay trong router.

Ghi raw từng step, bins tối đa100 slot, toàn bộ update loss/TD/gradient/epsilon; vẽ raw + trung bình trượt không vượt ranh giới curriculum. Validation chỉ có các mốc thực đo.

Dừng mềm trước hạn; lưu optimizer, target networks, replay, RNG và từng game đã hoàn tất. Không cam kết cả bốn run xong trong12 giờ. Dùng lại cùng runtime/GPU khi resume; đổi runtime sẽ bị chặn để tránh gọi nhầm là exact resume.

In [ ]:
from pathlib import Path
import time
SESSION_START = time.monotonic()
SOURCE_PATH = None
RESUME_INPUT_DIR = None  # tự tìm training_resume.zip / output cũ nếu chỉ có một bản
OUTPUT_ROOT = Path('/kaggle/working/bs5_v4_training')
DEVICE = 'auto'  # GPU nếu có; resume dùng device đã khóa
MAX_TOTAL_SESSION_SECONDS = 10.5 * 3600
EXPORT_RESERVE_SECONDS = 30 * 60
RUN_MAIN = True

In [ ]:
import os, sys, json, zipfile, hashlib, shutil, subprocess, time
from pathlib import Path
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
sys.dont_write_bytecode = True
if 'SESSION_START' not in globals(): SESSION_START = time.monotonic()
INPUT, WORK = Path('/kaggle/input'), Path('/kaggle/working')
WORK.mkdir(parents=True, exist_ok=True)
MARKER = 'configs/bs5_return_v4_fresh_train.json'
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()
def is_main_archive(path, version):
    try:
        with zipfile.ZipFile(path) as z:
            names=[n for n in z.namelist() if len(Path(n).parts)==2 and Path(n).name=='protocol.json']
            if len(names)!=1: return False
            p=json.loads(z.read(names[0]))
            return p.get('version')==version and not p.get('diagnostic',False)
    except (OSError,ValueError,zipfile.BadZipFile): return False
def extract_safe(archive, target):
    target=Path(target).resolve()
    for m in archive.infolist():
        dest=(target/m.filename).resolve()
        if not dest.is_relative_to(target) or (m.external_attr>>16)&0o170000==0o120000:
            raise RuntimeError('Unsafe ZIP member: '+m.filename)
    if archive.testzip() is not None: raise RuntimeError('ZIP CRC failed')
    archive.extractall(target)
if SOURCE_PATH is None:
    choices=[]
    for m in INPUT.rglob('DATASET_MANIFEST.json'):
        if (m.parent/MARKER).exists(): choices.append(m.parent)
    for f in INPUT.rglob('*.zip'):
        try:
            with zipfile.ZipFile(f) as z:
                if 'uav_thesis/DATASET_MANIFEST.json' in z.namelist() and 'uav_thesis/'+MARKER in z.namelist(): choices.append(f)
        except zipfile.BadZipFile: pass
    # Identical copies are safe; different versions require an explicit path.
    identities={}
    for f in choices:
        if f.is_dir(): raw=(f/'DATASET_MANIFEST.json').read_bytes()
        else:
            with zipfile.ZipFile(f) as z: raw=z.read('uav_thesis/DATASET_MANIFEST.json')
        identities.setdefault(hashlib.sha256(raw).hexdigest(),[]).append(f)
    if len(identities)!=1: raise RuntimeError('Add Input đúng source Dataset; hoặc điền SOURCE_PATH: '+str(choices))
    SOURCE_PATH=next(iter(identities.values()))[0]
SOURCE_PATH=Path(SOURCE_PATH)
SOURCE_ROOT=Path('/kaggle/temp/bs5_v4_fresh_source/uav_thesis')
if SOURCE_PATH.is_dir(): expected=(SOURCE_PATH/'DATASET_MANIFEST.json').read_bytes()
else:
    with zipfile.ZipFile(SOURCE_PATH) as z: expected=z.read('uav_thesis/DATASET_MANIFEST.json')
if SOURCE_ROOT.exists():
    if (SOURCE_ROOT/'DATASET_MANIFEST.json').read_bytes()!=expected: raise RuntimeError('Working source khác Dataset. Giữ output và dùng phiên sạch.')
else:
    SOURCE_ROOT.parent.mkdir(parents=True,exist_ok=True)
    if SOURCE_PATH.is_dir(): shutil.copytree(SOURCE_PATH,SOURCE_ROOT)
    else:
        with zipfile.ZipFile(SOURCE_PATH) as z: extract_safe(z,SOURCE_ROOT.parent)
manifest=json.loads(expected)
for rel,digest in manifest.items():
    if sha(SOURCE_ROOT/rel)!=digest: raise RuntimeError('Source checksum failed: '+rel)
RUN_ENV=os.environ.copy()
RUN_ENV.update(PYTHONDONTWRITEBYTECODE='1',OMP_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1',MKL_NUM_THREADS='1',CUBLAS_WORKSPACE_CONFIG=':4096:8')
RUN_ENV['PYTHONPATH']=str(SOURCE_ROOT)
probe="import torch,numpy,matplotlib,json; print(json.dumps(dict(torch=torch.__version__,numpy=numpy.__version__,matplotlib=matplotlib.__version__,cuda=torch.cuda.is_available())))"
check=subprocess.run([sys.executable,'-B','-c',probe],cwd=SOURCE_ROOT,env=RUN_ENV,capture_output=True,text=True)
if check.returncode: raise RuntimeError('Kaggle runtime cần torch/numpy/matplotlib. Không tự thay runtime khi resume. '+check.stderr[-2000:])
print('Verified source:',SOURCE_ROOT,'files:',len(manifest))
print('Installed runtime:',check.stdout.strip())
def run_stream(args, log):
    log=Path(log);log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('a',encoding='utf-8') as f:
        proc=subprocess.Popen([sys.executable,'-B','-u',*map(str,args)],cwd=SOURCE_ROOT,env=RUN_ENV,
                              stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in proc.stdout: print(line,end='',flush=True);f.write(line);f.flush()
        return proc.wait()
def restore_archive_or_directory(src, destination, required):
    src,destination=Path(src),Path(destination)
    if destination.exists(): raise RuntimeError('Không chép đè output hiện có: '+str(destination))
    if src.is_dir() and (src/'protocol.json').exists():
        shutil.copytree(src,destination)
    else:
        if src.is_dir():
            candidates=list(src.rglob(required))
            if len(candidates)!=1: raise RuntimeError('Chọn đúng thư mục hoặc ZIP '+required+': '+str(candidates))
            src=candidates[0]
        with zipfile.ZipFile(src) as z:
            # Transport archives have exactly one top-level run directory.
            roots={Path(n).parts[0] for n in z.namelist() if Path(n).parts}
            if len(roots)!=1: raise RuntimeError('Resume ZIP phải có một thư mục gốc.')
            top=next(iter(roots))
            staging=WORK/('restore_'+destination.name)
            if staging.exists(): raise RuntimeError('Restore dở dang; giữ thư mục để kiểm tra: '+str(staging))
            staging.mkdir()
            extract_safe(z,staging)
            shutil.move(str(staging/top),str(destination));staging.rmdir()
    for checksum_name in ['EXPORT_SHA256SUMS.json','HANDOFF_SHA256SUMS.json']:
        checksum=destination/checksum_name
        if checksum.exists():
            for rel,digest in json.loads(checksum.read_text()).items():
                if sha(destination/rel)!=digest: raise RuntimeError('Input checksum failed: '+rel)
    if not (destination/'protocol.json').exists(): raise RuntimeError('Thiếu protocol trong resume.')
def show_exports(root):
    from IPython.display import display, FileLink, Markdown
    exports=sorted([p for p in WORK.glob(root.name+'_export*') if (p/'EXPORT_INDEX.json').exists()],key=lambda p:p.stat().st_mtime)
    if not exports: raise RuntimeError('Không có gói export hoàn tất; giữ nguyên output và log.')
    export=exports[-1]
    report=export/'analysis/REPORT.md'
    if report.exists(): display(Markdown(report.read_text()))
    for f in sorted(export.iterdir()):
        if f.is_file(): display(FileLink(str(f)))
    print('OUTPUT:',export)
    if json.loads((export/'EXPORT_INDEX.json').read_text()).get('export_audit')=='failed': raise RuntimeError('Export audit failed; raw evidence archive has been saved above.')
    return export

In [ ]:
if RESUME_INPUT_DIR is None and not OUTPUT_ROOT.exists():
    candidates=[x for x in INPUT.rglob('training_resume.zip') if is_main_archive(x,'bs5_return_v4_fresh_crcn_v1')]
    directories=[]
    for f in INPUT.rglob('protocol.json'):
        try:
            p=json.loads(f.read_text())
            if p.get('version')=='bs5_return_v4_fresh_crcn_v1' and not p.get('diagnostic',False) and list((f.parent/'runs').glob('*/resume/*.pt')): directories.append(f.parent)
        except (ValueError,OSError): pass
    choices=candidates if candidates else directories
    if len(choices)>1: raise RuntimeError('Có nhiều bản resume; điền RESUME_INPUT_DIR rõ ràng.')
    if choices: RESUME_INPUT_DIR=choices[0]
if RESUME_INPUT_DIR is not None and not OUTPUT_ROOT.exists(): restore_archive_or_directory(RESUME_INPUT_DIR,OUTPUT_ROOT,'training_resume.zip')
PREFLIGHT=WORK/'bs5_v4_fresh_preflight'
rc=run_stream(['scripts/verification/preflight_bs5_v4_fresh.py','--output',PREFLIGHT],WORK/'bs5_v4_preflight.log')
if rc: raise RuntimeError('Preflight chưa PASS; main không chạy. Giữ nguyên log/receipt để kiểm tra.')
receipt=json.loads((PREFLIGHT/'receipt.json').read_text())
assert receipt['status']=='passed'
print('Preflight PASS:',receipt['tests'],'tests; diagnostic seconds:',round(receipt['wall_s'],1))

In [ ]:
MAIN_RETURN_CODE=0
if RUN_MAIN:
    if not OUTPUT_ROOT.exists():
        rc=run_stream(['scripts/training/run_bs5_v4_fresh_train.py','prepare','--output',OUTPUT_ROOT,'--device',DEVICE],WORK/'bs5_v4_prepare.log')
        if rc: raise RuntimeError('Prepare failed; giữ log.')
    preflight=OUTPUT_ROOT/'preflight';preflight.mkdir(exist_ok=True)
    shutil.copy2(PREFLIGHT/'receipt.json',preflight/'receipt.json')
    shutil.copy2(WORK/'bs5_v4_preflight.log',preflight/'preflight.log')
    budget=max(0,MAX_TOTAL_SESSION_SECONDS-EXPORT_RESERVE_SECONDS-(time.monotonic()-SESSION_START))
    print('Main soft budget remaining:',round(budget/3600,2),'hours')
    MAIN_RETURN_CODE=run_stream(['scripts/training/run_bs5_v4_fresh_train.py','run','--output',OUTPUT_ROOT,'--max-seconds',str(budget)],WORK/'bs5_v4_training.log')
    print(json.loads((OUTPUT_ROOT/'status.json').read_text()))
else: print('RUN_MAIN=False: bounded diagnostic preflight only.')

In [ ]:
if RUN_MAIN and (OUTPUT_ROOT/'protocol.json').exists():
    rc=run_stream(['scripts/training/run_bs5_v4_fresh_train.py','export','--output',OUTPUT_ROOT],WORK/'bs5_v4_export.log')
    EXPORT=show_exports(OUTPUT_ROOT)
    if rc: raise RuntimeError('Export audit failed; raw output vẫn được giữ. Xem log trước khi tiếp tục.')
    if MAIN_RETURN_CODE: raise RuntimeError('Main có lỗi; gói bằng chứng đã xuất. Không xóa/sửa ledger để chạy lại.')

## Kết quả và tiếp tục

- **review_small.zip**: báo cáo, learning curve PNG/PDF, bảng bins/episodes/validation, toàn bộ update diagnostics và cycle records.
- **training_resume.zip**: giữ đầy đủ để tiếp tục training nếu trạng thái `paused_at_game_boundary`. Add Input ZIP này hoặc dùng output notebook phiên trước; Run All sẽ nhận tiến độ. Cần cùng source/runtime/GPU.
- **training_for_benchmark.zip**: chỉ xuất khi đủ cả bốn run và mọi validation. Đây là gói nhỏ chứa checkpoint best/last cùng lựa chọn đã khóa; dùng notebook benchmark riêng. Không dùng gói này để resume training.
- `partial_error`: giữ nguyên tất cả output. Nếu có game bị ngắt đột ngột trước commit, ledger sẽ chặn lặp compute âm thầm; không reset counter. Dừng mềm thì resume được bình thường.
- Chỉ lưu file trong session chưa bảo đảm đã giữ qua phiên khác: hãy lưu notebook version với output. Không bấm chạy mới từ đầu khi muốn resume.
- Các seed test260001–260010 đã khóa trong benchmark_plan.json nhưng **không chạy** ở đây. Seed551/552 là seed train, khác seed tình huống test.
- S1/S2 dùng fixture dạy phối hợp Relay; đây là ablation cùng recipe, không phải curriculum được tối ưu riêng cho C-N. Hai seed là kiểm tra lặp lại còn hạn chế.